# SOLA 4237 Assignment 2 - Household Battery Energy Storage System (BESS) Optimisation

### Site: `23W0158` - `NSW Thredbo, Alpine` - `2022` / `2023`

### SOLA4237 Consumer Energy Resources (CER) Operations and Integration

<div style="font-size: 1rem; font-weight: normal; line-height: 1.4;">
Student: <b>Bilal Parkar</b> &nbsp;|&nbsp; zID: <b>z5476566</b><br>
Prepared for: VPP-SPREE single-site optimisation division (retailer client review)
</div>

---

### Purpose and Context
This notebook holds **all analysis and optimisation code** behind the Assignment 2 report. The report must show that VPP-SPREE's single-site optimisation can deliver real, quantified benefit to a household across **flat, time-of-use (ToU) and wholesale pass-through tariffs**, and that it still works when **PV and price forecasts are imperfect**.

| # | Question the report must answer | Evidence built in |
|---|---|---|
| 1 | What are the site, data quality and tariff context? | Parts 1-4 |
| 2 | What battery (size, power, efficiency, limits, degradation) suits this site? | Parts 4, 5, 8 |
| 3 | What is the benchmark (no battery, simple self-consumption)? | Part 6 |
| 4 | What is the best possible operation on each tariff (perfect foresight)? | Part 7 |
| 5 | How good are our PV and price forecasts? | Parts 9, 10 |
| 6 | How much value is lost to uncertainty, and is PV or price the bigger driver? | Parts 11, 12 |
| 7 | Annual bill, self-consumption, savings, payback, VPP fee vs value | Part 13 |

### How this notebook is organised
- **Part 0** setup and constants. **Parts 1-3** load, check and prepare data. **Part 4** characterises the site. **Part 5** is the battery model. **Part 6** benchmarks. **Part 7** perfect-foresight optimiser. **Part 8** sizing and payback. **Parts 9-10** forecasts. **Part 11** 48-hour rolling-horizon scheduler with 5-minute execution. **Part 12** perfect vs uncertain comparison. **Part 13** techno-economics. **Part 14** discussion and recommendations. **Part 15** report figures. **Appendix A** AI disclosure and verification log. **Appendix B** reproducibility.
- Each task follows one pattern: *markdown explanation -> function -> test -> apply to site data -> findings cell*.
- Functions are tested on tiny synthetic samples (`assert`) before they touch site data.
- **Findings cells** (blockquotes) record what the data showed. Anything marked `TODO` is not finished.
- Set `SOLA_DATA_PATH` (or edit `DATA_PATH` in Part 0) and run top to bottom. Set `QUICK_RUN_DAYS` in Part 0 for a fast dry run of the slow parts (11-12).

### Required files
- Assignment CSV (two years of 5-minute data). Column names are mapped in one place (`COL` in Part 0): **check these against your CSV first.**
- AEMO pre-dispatch price forecasts for the site's region (only if you choose them over your own price model; see Part 10).

### Contents and status

| Part | Title | Status | Note |
|---|---|---|---|
| 0 | Setup and configuration | Skeleton | Fill site metadata, tariff windows, cost assumptions |
| 1 | Data loading and integrity | Skeleton | Functions tested; run on site data |
| 2 | Data quality checks | Skeleton | Functions tested; record decisions |
| 3 | Feature engineering and tariffs | Skeleton | Wholesale retail price build needs sourced network/fee values |
| 4 | Site characterisation | Skeleton | Load vs PV, shiftable energy, topology assumption |
| 5 | Battery model | Skeleton | Choose real product spec; temperature derating needs a source |
| 6 | Benchmarks and billing | Skeleton | No battery, self-consumption |
| 7 | Perfect-foresight optimiser | Skeleton | LP written and tested; run on 3 tariffs |
| 8 | Sizing and payback | Skeleton | Capex assumptions need sources |
| 9 | PV forecasting | Skeleton | Own model vs commercial forecast |
| 10 | Price forecasting | Skeleton | AEMO pre-dispatch or own model |
| 11 | Rolling-horizon scheduler | Skeleton | 48 h hourly plan, 5 min execution |
| 12 | Perfect vs uncertain; PV vs price | Skeleton | Ablation study |
| 13 | Techno-economics and metrics | Skeleton | Master results table |
| 14 | Discussion and recommendations | Skeleton | Fill after results |
| 15 | Report figures | Skeleton | Planned figure list |
| A | AI disclosure and verification log | Skeleton | Fill as you work, not at the end |
| B | Reproducibility | Skeleton | Versions, seed, runtime |

#### Brief requirements tracker

| # | Requirement from the brief | Where | Status |
|---|---|---|---|
| 1 | Default self-consumption mode as benchmark | Part 6 | TODO |
| 2 | AC vs DC bus assumption, losses | Parts 4, 5 | TODO |
| 3 | BESS size, power, SOC limits, temperature limits, degradation cost | Parts 5, 8 | TODO |
| 4 | Optimal strategy for flat, ToU, wholesale | Part 7 | TODO |
| 5 | Wholesale retail price (network, fees, certificates, hedging) | Part 3 | TODO |
| 6 | $25/month VPP fee on wholesale pass-through | Part 13 | TODO |
| 7 | Zero export (curtail) when prices are negative | Part 7 | TODO |
| 8 | PV forecast model trained on Year 1, compared with commercial forecast (MAE, RMSE, nRMSE) | Part 9 | TODO |
| 9 | Price forecast (own model or AEMO pre-dispatch) | Part 10 | TODO |
| 10 | 48 h rolling horizon, hourly replanning, 5-minute execution without knowing current-interval price/solar | Part 11 | TODO |
| 11 | Settle on actual price, BESS flows, load and solar | Part 11 | TODO |
| 12 | Uncertainty vs perfect foresight; PV vs price influence | Part 12 | TODO |
| 13 | Annual bill, self-consumption, savings, payback | Part 13 | TODO |
| 14 | AI disclosure with evidence of checking AI output | Appendix A | TODO |

# Part 0 - Setup and Configuration

### Conventions used throughout (verify in Part 2)
- **Battery power** `batt_kW`: **positive = discharging**, **negative = charging** (same convention as Assignment 1).
- **Grid power** `grid_kW`: **positive = import**, **negative = export**.
- **Accounting identity:** `grid = load - (solar - curtailed) - battery`.
- **Power to energy:** `kWh = kW x dt_h` (`dt_h = 5/60` for raw data, `1` for hourly plans).
- **Prices** are in **$/kWh** inside the code. Wholesale data arrives in $/MWh, so divide by 1000.
- **Time:** timestamps are naive local time. State whether they are NEM time (fixed AEST, UTC+10) in Part 1 and align prices accordingly.

In [7]:
import os, io, warnings, dataclasses
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import linprog
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [8]:
# ---------- Paths ----------
DATA_PATH = os.environ.get(
    "SOLA_DATA_PATH",
    "/Users/bilalp/Documents/SOLA4237/CER_Assignments/Assignment_2/SOLA/Data_NSW_Thredbo - Alpine_2022-2023_partial.csv",
)
# ---------- Column map: the ONLY place raw CSV names appear ----------
COL = dict(
    time     = "timestamp",
    load     = "load_kW",
    solar    = "solar_output_kW",                        # measured PV (expected Year 1 only)
    solar_fc = "commercial_forecasted_solar_output_kW",  # commercial forecast (expected Year 2 only)
    irr      = "irradiance_W/m^2",
    temp     = "temperature_C",
    rrp      = "spot_price_AUD/kWh",                     # already $/kWh: do NOT divide by 1000
    flat_imp = "tariff_import_flat_AUD/kWh",
    flat_exp = "tariff_export_flat_AUD/kWh",
    tou_imp  = "tariff_import_tou_AUD/kWh",
    tou_exp  = "tariff_export_tou_AUD/kWh",
)

# ---------- Site metadata (from CER-data-generator screenshot) ----------
SITE_ID      = "NSW_Thredbo"
LOCATION     = "Thredbo, NSW (Alpine)"
NEM_REGION   = "NSW1"
LOAD_PROFILE = "23W0158 (scaled x29.1)"
PV_PR        = 0.85      # performance ratio
PV_GAMMA     = -0.004    # power temperature coefficient, per degC

# ---------- Time ----------
INTERVAL_MIN   = 5
DT_H           = INTERVAL_MIN / 60      # hours per interval: kWh = kW * DT_H
STEPS_PER_HOUR = 60 // INTERVAL_MIN

# Part 1 - Data Loading and Integrity

### Background: why check the timeline before anything else?
Every later calculation assumes one row per 5-minute interval. A missing or duplicated interval silently biases energy totals, and a clock shift (daylight saving) misaligns solar noon and price merges. "No nulls" is not the same as "complete". This dataset also mixes **two years with different columns populated** (measured solar in Year 1, commercial forecast in Year 2), so nulls in those columns are expected, not faults.

## Task 1.1 - Load the telemetry

In [9]:
def load_telemetry(path) -> pd.DataFrame:
    out = pd.read_csv(path, parse_dates=["timestamp"])
    return out.set_index("timestamp").sort_index()

In [10]:
"""Tests for Task 1.1"""
_csv = io.StringIO("timestamp,x\n2024-01-01 00:05:00,2\n2024-01-01 00:00:00,1\n")
_t = load_telemetry(_csv)
assert isinstance(_t.index, pd.DatetimeIndex)   # real datetimes, not strings
assert _t.index.is_monotonic_increasing          # sorted
assert _t["x"].tolist() == [1, 2]                # values travelled with their timestamps
print("Task 1.1 tests passed")

Task 1.1 tests passed


In [11]:
df = load_telemetry(DATA_PATH)
print(df.shape)
df.head()

(210240, 10)


,irradiance_W/m^2,temperature_C,solar_output_kW,commercial_forecasted_solar_output_kW,load_kW,spot_price_AUD/kWh,tariff_import_flat_AUD/kWh,tariff_export_flat_AUD/kWh,tariff_import_tou_AUD/kWh,tariff_export_tou_AUD/kWh
timestamp,,,,,,,,,,
2022-01-01 00:00:00,0.0,15.1,0.0,NaN,0.53,0.12477,0.3,0.05,0.15,0.05
2022-01-01 00:05:00,0.0,15.1,0.0,NaN,0.54,0.12486,0.3,0.05,0.15,0.05
2022-01-01 00:10:00,0.0,15.1,0.0,NaN,0.53,0.12602,0.3,0.05,0.15,0.05
2022-01-01 00:15:00,0.0,15.0,0.0,NaN,0.51,0.12493,0.3,0.05,0.15,0.05
2022-01-01 00:20:00,0.0,15.0,0.0,NaN,0.52,0.11354,0.3,0.05,0.15,0.05
